In [ ]:
%matplotlib inline

# DOE


In [ ]:
from __future__ import annotations

from gemseo.discipline import AnalyticDiscipline
from gemseo.doe import PYDOE_FULLFACT_Settings
from gemseo.post import OptHistoryView_Settings
from gemseo.space import DesignSpace
from gemseo.space import RandomSpace
from gemseo.uncertainty.distribution import OTNormalDistribution_Settings

from gemseo_umdo.formulations.sampling_settings import Sampling_Settings
from gemseo_umdo.scenarios.udoe_scenario import UDOEScenario

Firstly,
we define an [AnalyticDiscipline][gemseo.discipline.analytic.AnalyticDiscipline]
implementing the random function $f(x,U)=(x+U)^2$:



In [ ]:
discipline = AnalyticDiscipline({"y": "(x+u)**2"}, name="quadratic_function")

where $x$ belongs to the interval $[-1,1]$:



In [ ]:
design_space = DesignSpace()
design_space.add_variable("x", lower_bound=-1, upper_bound=1.0, value=0.5)

and $U$ is a standard Gaussian variable:



In [ ]:
uncertain_space = RandomSpace()
uncertain_space.add_variable("u", OTNormalDistribution_Settings())

Then,
we build a [UDOEScenario][gemseo_umdo.scenarios.udoe_scenario.UDOEScenario]
to minimize a sampling-based estimation
of the expectation $\mathbb{E}[Y]$ where $Y=f(x,U)$:



In [ ]:
scenario = UDOEScenario(
    [discipline],
    design_space,
    uncertain_space,
    Sampling_Settings(n_samples=100),
)
scenario.add_objective("y", "Mean")

We execute it with a full-factorial design of experiments:



In [ ]:
scenario.execute(PYDOE_FULLFACT_Settings(n_samples=100))

and plot the history:



In [ ]:
scenario.post_process(OptHistoryView_Settings(save=True, show=True))

Notice that the numerical solution is close to $(x^*,f^*)=(0,1)$ as expected
from the expression of the statistic: $\mathbb{E}[Y]=\mathbb{E}[(x+U)^2]=x^2+1$.

